# 02 · Feature extraction
Reproduce canonical features and approved masked training statistics using frozen train/validation manifests.
By default outputs are isolated in temporary storage; installed artifacts and the test split are not modified. Writers and loaders share the canonical cache schema and provenance checks. The complete closeout call is completed by correction package W12.


In [ ]:
from pathlib import Path
import sys,json,hashlib,tempfile,numpy as np,pandas as pd,yaml
PROJECT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT/'src'))
from dronedet import features
from dronedet.config import load_config
from dronedet.data_preparation import locate_dataset_root
PROJECT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
CFG=yaml.safe_load((PROJECT/'config.yaml').read_text(encoding='utf-8'))
profile='colab' if 'google.colab' in sys.modules else 'local'; P={k:Path(v) for k,v in CFG['paths'][profile].items()}
def safe(path):
 path=Path(path).resolve(); raw=P['raw_root'].resolve()
 if path==raw or raw in path.parents: raise PermissionError(path)
 path.parent.mkdir(parents=True,exist_ok=True); return path
fe=load_config(PROJECT/'config.yaml'); root=locate_dataset_root(P['raw_root'])
hashes=json.loads((PROJECT/'results/tables/manifest_hashes.json').read_text(encoding='utf-8'))
BUILD_SPLITS=('train','val')
_output_directory=tempfile.TemporaryDirectory(prefix='dronedet_features_')
OUTPUT_ROOT=Path(_output_directory.name)
OUTPUT_PATHS={'root':PROJECT,'splits':PROJECT/'data/processed/splits','spec':OUTPUT_ROOT/'data/processed/spectrograms','tab':OUTPUT_ROOT/'data/processed/features_tabular','raw_root':P['raw_root']}
print('Isolated output:', OUTPUT_ROOT)
for s in BUILD_SPLITS:
 assert hashlib.sha256((PROJECT/'data/processed/splits'/f'{s}.csv').read_bytes()).hexdigest()==hashes[s], f'{s} split changed after freeze'
print(fe.validate_filterbank(), 'n_win=',fe.n_win,'win_samples=',fe.win_samples)

In [ ]:
summaries={}; train_cache=train_index=None
for split in BUILD_SPLITS:
 df=pd.read_csv(PROJECT/'data/processed/splits'/f'{split}.csv',keep_default_na=False)
 features.write_split_cache(df,fe,root,OUTPUT_PATHS,split=split)
 cache=features.load_power_cache(OUTPUT_PATHS,split,mmap_mode=None)
 index=features.load_cache_index(OUTPUT_PATHS,split)
 X=features.load_tabular(OUTPUT_PATHS,split)['X']
 assert len(index)==len(df)==len(X) and X.shape[1]==len(features.feature_names()) and np.isfinite(X).all()
 summaries[split]={'clips':len(df),'mel_shape':list(cache.shape),'features_shape':list(X.shape),'silent':int(index.silent.sum()),'resampled':int((index.length_status!='unchanged').sum()),'crop_all_zero':int(index.crop_all_zero.sum())}
 if split=='train': train_cache,train_index,train_manifest=cache,index,df
 print(split,summaries[split])
assert sum(v['clips'] for v in summaries.values()) == sum(len(pd.read_csv(PROJECT/'data/processed/splits'/f'{s}.csv')) for s in BUILD_SPLITS)
normalizer=features.fit_cached_normalizer(train_cache,train_index,train_manifest.to_dict('records'),fe)
mu,sigma=normalizer.mean,normalizer.std
frame_count=normalizer.fit_record['frame_count']
assert mu.shape==(fe.n_mels,) and sigma.shape==(fe.n_mels,) and np.isfinite(mu).all() and np.isfinite(sigma).all() and (sigma>0).all()
np.savez_compressed(safe(OUTPUT_ROOT/'data/processed/spectrograms/norm_stats_global.npz'),mean=mu.astype(np.float32),std=sigma.astype(np.float32),frame_count=np.int64(frame_count),clips_used=np.int64(train_index.stats_eligible.sum()),clips_excluded=np.int64((~train_index.stats_eligible).sum()))
(safe(OUTPUT_ROOT/'data/processed/spectrograms/normalization_fit.json')).write_text(json.dumps(normalizer.fit_record,indent=2)+'\n',encoding='utf-8')
print('train-only global stats:',frame_count,'frames;',int(train_index.stats_eligible.sum()),'eligible clips')
print('Decision log draft: Tabular descriptors use 19 MFCCs without c0, HTK mel centres from edge frequencies, rolloff 0.85, population standard deviation, and power=1 flatness; mel cache is full-clip and all centering uses frame-aligned offsets.')

In [ ]:
# Check report-filterbank parity using the owner's implementation; save once and never regenerate a golden.
from dronedet.config import FrontEndConfig
from dronedet.dsp import mel_filterbank
reference=FrontEndConfig.reference()
ours=mel_filterbank(reference.sr,reference.n_fft,reference.n_mels,reference.fmin,reference.fmax)
lib=__import__('librosa').filters.mel(sr=reference.sr,n_fft=reference.n_fft,n_mels=reference.n_mels,fmin=reference.fmin,fmax=reference.fmax,htk=True,norm='slaney',dtype=np.float32)
support=np.array_equal(ours>0,lib>0)
relative=float(np.max(np.abs(ours[lib>0]-lib[lib>0])/np.abs(lib[lib>0])))
assert support and relative<=1e-6
golden_path=safe(OUTPUT_ROOT/'data/processed/golden/report_filterbank.npy')
if golden_path.exists(): assert np.array_equal(np.load(golden_path,allow_pickle=False),ours)
else:
 with golden_path.open('xb') as stream: np.save(stream,ours,allow_pickle=False)
print('report filterbank parity: identical support; max relative error',relative)
assert set(summaries)==set(BUILD_SPLITS)
assert all(v['features_shape'][1]==84 for v in summaries.values())
for split in BUILD_SPLITS:
 df=pd.read_csv(PROJECT/'data/processed/splits'/f'{split}.csv',keep_default_na=False)
 idx=pd.read_csv(OUTPUT_ROOT/'data/processed/spectrograms'/f'index_{split}.csv',keep_default_na=False)
 assert len(df)==len(idx) and idx.clip_id.equals(df.clip_id)
edge_checks=features.run_edge_case_checks(fe)
assert all(case['pass'] for case in edge_checks.values()), edge_checks
print('Executed edge checks:',len(edge_checks))
print('Executed filterbank comparison and edge cases; complete native-rate bundle checks are a separate gate.')
print('Decision log draft: Tabular descriptors use 19 MFCCs without c0, HTK mel centres from edge frequencies, rolloff 0.85, population standard deviation, and power=1 flatness; mel cache is full-clip and all centering uses frame-aligned offsets.')